Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved. SPDX-License-Identifier: Apache-2.0

# AWS for SAP MCP Server as AgentCore Gateway Target

## Overview

This notebook walks through adding the [AWS for SAP MCP Server](https://docs.aws.amazon.com/mcp-sap/latest/awsforsapmcp/introduction.html) as an MCP target on [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html). The SAP MCP Server provides AI agents with structured, protocol-driven access to SAP S/4HANA and SAP ECC OData V2 services.

| Information | Details |
|:---|:---|
| Target type | MCP Server Target |
| SAP MCP Server | [AWS for SAP MCP Server](https://docs.aws.amazon.com/mcp-sap/latest/awsforsapmcp/introduction.html) |
| Communication | Streamable HTTP (`/mcp` endpoint) |
| Outbound auth | CustomOauth2 (SAP MCP Server Cognito) |
| Inbound auth | Amazon Cognito (M2M) |
| Tools exposed | 9 (5 read + 4 write, read-only by default) |
| Default mode | Read-only (write operations disabled unless explicitly enabled) |

### Prerequisites

Before starting this notebook, you need:

1. **AWS account** with access to Amazon Bedrock AgentCore
2. **AWS for SAP MCP Server** deployed — see [Getting Started](https://docs.aws.amazon.com/mcp-sap/latest/awsforsapmcp/getting-started.html)
3. **SAP MCP Server endpoint URL** and **Cognito credentials** for M2M authentication
4. **Python 3.11+** with Jupyter kernel
5. **AWS CLI** configured with appropriate credentials

You can either create a new gateway here or reuse the one created in [01-salesforce-gateway-target.ipynb](01-salesforce-gateway-target.ipynb).

In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
import getpass
import json
import logging
import time
import uuid

import boto3
import requests
from boto3.session import Session

import gateway_mcp_client

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
)

session = Session()
REGION = session.region_name or "us-east-1"
print(f"Using region: {REGION}")

In [ ]:
# Collect SAP MCP Server credentials
SAP_MCP_ENDPOINT = input("Enter the SAP MCP Server endpoint URL (e.g., https://<runtime-url>/mcp): ")
SAP_TOKEN_ENDPOINT = input("Enter the SAP MCP Server Cognito token endpoint: ")
SAP_CLIENT_ID = input("Enter the SAP MCP Server Client ID: ")
SAP_CLIENT_SECRET = getpass.getpass("Enter the SAP MCP Server Client Secret: ")
SAP_SCOPE = input("Enter the OAuth scope (e.g., <resource-server-id>/invoke): ")

assert SAP_MCP_ENDPOINT.strip(), "SAP MCP endpoint cannot be empty"
assert SAP_TOKEN_ENDPOINT.strip(), "Token endpoint cannot be empty"
assert SAP_CLIENT_ID.strip(), "Client ID cannot be empty"
assert SAP_CLIENT_SECRET.strip(), "Client Secret cannot be empty"

print(f"\nSAP MCP endpoint: {SAP_MCP_ENDPOINT}")
print(f"Token endpoint: {SAP_TOKEN_ENDPOINT}")

## AWS for SAP MCP Server — Architecture

The [AWS for SAP MCP Server](https://docs.aws.amazon.com/mcp-sap/latest/awsforsapmcp/introduction.html) runs on Amazon Bedrock AgentCore Runtime and provides AI agents with access to SAP S/4HANA and SAP ECC OData V2 services.

**Key characteristics:**
- Communicates via **Streamable HTTP** on a `/mcp` endpoint
- **Read-only by default** — write operations require explicit opt-in
- Credentials are **never stored on disk** — retrieved at runtime from AWS Secrets Manager
- Supports **Basic Auth** or **OAuth 2.0** for outbound SAP authentication
- Deployed via **CloudFormation template**

### Available Tools

| Tool | Category | Description |
|---|---|---|
| `find_sap_services` | Read | Discover available SAP OData services from the catalog |
| `get_metadata` | Read | Retrieve OData service metadata (entity types, properties) |
| `get_service_hints` | Read | Get usage guidance for specific services |
| `odata_read` | Read | Read data from SAP OData entity sets with filtering |
| `odata_count` | Read | Count records in an entity set (use before reads) |
| `odata_create` | Write | Create new entity records (deep insert supported) |
| `odata_update` | Write | Update existing entity records |
| `odata_delete` | Write | Delete entity records |
| `odata_function_import` | Write | Execute custom backend logic |

> **Security:** Write tools are only registered when explicitly enabled in the SAP MCP Server configuration. Follow the principle of least privilege — only enable the specific write operations your use case requires.

REUSE_GATEWAY = input("Reuse an existing gateway? (yes/no): ").strip().lower() == "yes"

if REUSE_GATEWAY:
    GATEWAY_ID = input("Enter existing Gateway ID: ")
    GATEWAY_URL = input("Enter existing Gateway URL: ")
    GW_CLIENT_ID = input("Enter Cognito Client ID for the gateway: ")
    GW_CLIENT_SECRET = getpass.getpass("Enter Cognito Client Secret for the gateway: ")
    TOKEN_ENDPOINT = input("Enter Cognito token endpoint: ")
    FULL_SCOPE = input("Enter OAuth scope: ")
    CREATED_GATEWAY = False
    print(f"\nReusing gateway: {GATEWAY_ID}")
else:
    CREATED_GATEWAY = True
    GATEWAY_NAME = f"multi-isv-sap-tutorial-{str(uuid.uuid4())[:8]}"
    print(f"Creating new gateway: {GATEWAY_NAME}")

    # Create Cognito User Pool
    cognito_client = boto3.client("cognito-idp", region_name=REGION)
    pool_resp = cognito_client.create_user_pool(
        PoolName=f"{GATEWAY_NAME}-pool",
        Policies={"PasswordPolicy": {"MinimumLength": 8}},
    )
    USER_POOL_ID = pool_resp["UserPool"]["Id"]

    COGNITO_DOMAIN = f"{GATEWAY_NAME}-domain"
    cognito_client.create_user_pool_domain(Domain=COGNITO_DOMAIN, UserPoolId=USER_POOL_ID)
    TOKEN_ENDPOINT = f"https://{COGNITO_DOMAIN}.auth.{REGION}.amazoncognito.com/oauth2/token"

    RESOURCE_SERVER_ID = f"{GATEWAY_NAME}-id"
    SCOPE_NAME = "invoke"
    cognito_client.create_resource_server(
        UserPoolId=USER_POOL_ID,
        Identifier=RESOURCE_SERVER_ID,
        Name=f"{GATEWAY_NAME} Gateway",
        Scopes=[{"ScopeName": SCOPE_NAME, "ScopeDescription": "Invoke gateway"}],
    )
    FULL_SCOPE = f"{RESOURCE_SERVER_ID}/{SCOPE_NAME}"

    app_resp = cognito_client.create_user_pool_client(
        UserPoolId=USER_POOL_ID,
        ClientName=f"{GATEWAY_NAME}-client",
        GenerateSecret=True,
        AllowedOAuthFlows=["client_credentials"],
        AllowedOAuthScopes=[FULL_SCOPE],
        AllowedOAuthFlowsUserPoolClient=True,
    )
    GW_CLIENT_ID = app_resp["UserPoolClient"]["ClientId"]
    GW_CLIENT_SECRET = app_resp["UserPoolClient"]["ClientSecret"]
    DISCOVERY_URL = f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"

    # Create IAM role
    iam = boto3.client("iam")
    ROLE_NAME = f"agentcore-{GATEWAY_NAME}-role"
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }],
    }
    role_resp = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="IAM role for AgentCore Gateway SAP tutorial",
    )
    ROLE_ARN = role_resp["Role"]["Arn"]
    time.sleep(10)

    # Create gateway
    gw_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
    gw_resp = gw_client.create_gateway(
        name=GATEWAY_NAME,
        roleArn=ROLE_ARN,
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": DISCOVERY_URL,
                "allowedClients": [GW_CLIENT_ID],
            }
        },
    )
    GATEWAY_ID = gw_resp["gatewayId"]
    GATEWAY_URL = gw_resp["gatewayUrl"]

    print(f"Gateway created: {GATEWAY_ID}")
    print("Waiting for READY...")
    for _ in range(60):
        status = gw_client.get_gateway(gatewayIdentifier=GATEWAY_ID)["status"]
        if status == "READY":
            break
        time.sleep(5)
    print("✓ Gateway is READY")

gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

In [ ]:
# Test SAP MCP Server authentication
sap_token = gateway_mcp_client.get_cognito_m2m_token(
    token_endpoint=SAP_TOKEN_ENDPOINT,
    client_id=SAP_CLIENT_ID,
    client_secret=SAP_CLIENT_SECRET,
    scope=SAP_SCOPE,
)
print(f"✓ SAP MCP Server token obtained ({len(sap_token)} chars)")

## Step 2: Create Gateway (or Reuse Existing)

If you already have a gateway from the Salesforce notebook, you can skip this step and provide the existing gateway details.

In [ ]:
REUSE_GATEWAY = input("Reuse an existing gateway? (yes/no): ").strip().lower() == "yes"

if REUSE_GATEWAY:
    GATEWAY_ID = input("Enter existing Gateway ID: ")
    GATEWAY_URL = input("Enter existing Gateway URL: ")
    GW_CLIENT_ID = input("Enter Cognito Client ID for the gateway: ")
    GW_CLIENT_SECRET = getpass.getpass("Enter Cognito Client Secret for the gateway: ")
    TOKEN_ENDPOINT = input("Enter Cognito token endpoint: ")
    FULL_SCOPE = input("Enter OAuth scope: ")
    CREATED_GATEWAY = False
    print(f"\nReusing gateway: {GATEWAY_ID}")
else:
    CREATED_GATEWAY = True
    GATEWAY_NAME = f"multi-isv-sap-tutorial-{str(uuid.uuid4())[:8]}"
    print(f"Creating new gateway: {GATEWAY_NAME}")

    # Create Cognito User Pool
    cognito_client = boto3.client("cognito-idp", region_name=REGION)
    pool_resp = cognito_client.create_user_pool(
        PoolName=f"{GATEWAY_NAME}-pool",
        Policies={"PasswordPolicy": {"MinimumLength": 8}},
    )
    USER_POOL_ID = pool_resp["UserPool"]["Id"]

    COGNITO_DOMAIN = f"{GATEWAY_NAME}-domain"
    cognito_client.create_user_pool_domain(Domain=COGNITO_DOMAIN, UserPoolId=USER_POOL_ID)
    TOKEN_ENDPOINT = f"https://{COGNITO_DOMAIN}.auth.{REGION}.amazoncognito.com/oauth2/token"

    RESOURCE_SERVER_ID = f"{GATEWAY_NAME}-id"
    SCOPE_NAME = "invoke"
    cognito_client.create_resource_server(
        UserPoolId=USER_POOL_ID,
        Identifier=RESOURCE_SERVER_ID,
        Name=f"{GATEWAY_NAME} Gateway",
        Scopes=[{"ScopeName": SCOPE_NAME, "ScopeDescription": "Invoke gateway"}],
    )
    FULL_SCOPE = f"{RESOURCE_SERVER_ID}/{SCOPE_NAME}"

    app_resp = cognito_client.create_user_pool_client(
        UserPoolId=USER_POOL_ID,
        ClientName=f"{GATEWAY_NAME}-client",
        GenerateSecret=True,
        AllowedOAuthFlows=["client_credentials"],
        AllowedOAuthScopes=[FULL_SCOPE],
        AllowedOAuthFlowsUserPoolClient=True,
    )
    GW_CLIENT_ID = app_resp["UserPoolClient"]["ClientId"]
    GW_CLIENT_SECRET = app_resp["UserPoolClient"]["ClientSecret"]
    DISCOVERY_URL = f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"

    # Create IAM role
    iam = boto3.client("iam")
    ROLE_NAME = f"agentcore-{GATEWAY_NAME}-role"
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "gateway.bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }],
    }
    role_resp = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="IAM role for AgentCore Gateway SAP tutorial",
    )
    ROLE_ARN = role_resp["Role"]["Arn"]
    time.sleep(10)

    # Create gateway
    gw_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
    gw_resp = gw_client.create_gateway(
        name=GATEWAY_NAME,
        roleArn=ROLE_ARN,
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": DISCOVERY_URL,
                "allowedAudiences": [GW_CLIENT_ID],
                "allowedClients": [GW_CLIENT_ID],
            }
        },
    )
    GATEWAY_ID = gw_resp["gatewayId"]
    GATEWAY_URL = gw_resp["gatewayUrl"]

    print(f"Gateway created: {GATEWAY_ID}")
    print("Waiting for ACTIVE...")
    while True:
        status = gw_client.get_gateway(gatewayIdentifier=GATEWAY_ID)["status"]
        if status == "ACTIVE":
            break
        time.sleep(5)
    print("✓ Gateway is ACTIVE")

gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

SAP_TARGET_NAME = "sap-target"

target_resp = gateway_client.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name=SAP_TARGET_NAME,
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": SAP_MCP_ENDPOINT,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": SAP_CREDENTIAL_PROVIDER_ARN,
                    "scopes": [],
                    "grantType": "CLIENT_CREDENTIALS",
                }
            },
        }
    ],
)

SAP_TARGET_ID = target_resp["targetId"]
print(f"Created SAP target: {SAP_TARGET_NAME} ({SAP_TARGET_ID})")
print("Waiting for target to reach READY status...")

In [ ]:
SAP_CREDENTIAL_PROVIDER_NAME = f"sap-mcp-oauth-{str(uuid.uuid4())[:8]}"

# Extract Cognito discovery URL from token endpoint
# Token endpoint format: https://<domain>.auth.<region>.amazoncognito.com/oauth2/token
SAP_DISCOVERY_URL = input(
    "Enter the SAP MCP Server Cognito discovery URL\n"
    "(e.g., https://cognito-idp.<region>.amazonaws.com/<pool-id>/.well-known/openid-configuration): "
)

cred_resp = gateway_client.create_oauth2_credential_provider(
    name=SAP_CREDENTIAL_PROVIDER_NAME,
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "clientId": SAP_CLIENT_ID,
            "clientSecret": SAP_CLIENT_SECRET,
            "oauthDiscovery": {
                "discoveryUrl": SAP_DISCOVERY_URL,
            },
        }
    },
)

SAP_CREDENTIAL_PROVIDER_ARN = cred_resp["credentialProviderArn"]
print(f"Created SAP credential provider: {SAP_CREDENTIAL_PROVIDER_NAME}")
print(f"ARN: {SAP_CREDENTIAL_PROVIDER_ARN}")

## Step 4: Add SAP MCP Server as Gateway Target

We add the SAP MCP Server as an MCP server target on the gateway. This tells the gateway to proxy MCP requests to the SAP MCP Server endpoint.

In [ ]:
SAP_TARGET_NAME = "sap-target"

target_resp = gateway_client.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name=SAP_TARGET_NAME,
    targetConfiguration={
        "mcpTarget": {
            "mcpServer": {
                "endpoint": SAP_MCP_ENDPOINT,
            },
            "authConfiguration": {
                "oauth2Auth": {
                    "credentialProviderArn": SAP_CREDENTIAL_PROVIDER_ARN,
                }
            },
        }
    },
)

SAP_TARGET_ID = target_resp["targetId"]
print(f"Created SAP target: {SAP_TARGET_NAME} ({SAP_TARGET_ID})")
print("Waiting for target to reach READY status...")

In [ ]:
# Wait for target to become READY
gateway_mcp_client.wait_for_target_ready(
    client=gateway_client,
    gateway_id=GATEWAY_ID,
    target_name=SAP_TARGET_NAME,
    region=REGION,
    timeout=300,
)
print("\n✓ SAP MCP Server target is READY")

## Step 5: Verify SAP Tools via Gateway

Once the target is ready, the SAP MCP Server tools appear via the gateway's `tools/list` endpoint.

In [ ]:
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

mcp_client = MCPClient(
    lambda: streamablehttp_client(
        url=GATEWAY_URL,
        headers={
            "Authorization": f"Bearer {get_gw_token()}",
            "MCP-Protocol-Version": "2025-03-26",
        },
    )
)

SYSTEM_PROMPT = (
    "You are a helpful assistant with access to SAP tools via the AWS for SAP MCP Server. "
    "Before reading data, use get_metadata to understand available entity sets and field names. "
    "Use odata_count before odata_read to understand data volume. "
    "SAP field names can be non-obvious — always check metadata first."
)

with mcp_client:
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-6-v1",
        system_prompt=SYSTEM_PROMPT,
        tools=mcp_client.list_tools_sync(),
    )
    result = agent("What SAP services are available for sales orders? Show me the first 3 sales orders.")
    print(result)

## Step 6: Invoke SAP Tools

Let's call the SAP MCP Server tools through the gateway to explore available services and read data.

In [ ]:
# Discover available SAP services
result = mcp.call_tool(
    "sap-target___find_sap_services",
    {"search_term": "sales", "top": 5},
)
print("=== Find SAP Services (sales) ===")
content = result.get("result", {}).get("content", [])
for item in content:
    if item.get("type") == "text":
        print(item["text"][:2000])

In [ ]:
# Get metadata for Business Partner API
result = mcp.call_tool(
    "sap-target___get_metadata",
    {"service_name": "API_BUSINESS_PARTNER"},
)
print("=== Business Partner API Metadata ===")
content = result.get("result", {}).get("content", [])
for item in content:
    if item.get("type") == "text":
        print(item["text"][:3000])

In [ ]:
# Read business partners
result = mcp.call_tool(
    "sap-target___odata_read",
    {
        "service_name": "API_BUSINESS_PARTNER",
        "entity_set": "A_BusinessPartner",
        "top": 5,
        "select": "BusinessPartner,BusinessPartnerFullName,BusinessPartnerCategory",
    },
)
print("=== Read Business Partners ===")
content = result.get("result", {}).get("content", [])
for item in content:
    if item.get("type") == "text":
        try:
            data = json.loads(item["text"])
            print(json.dumps(data, indent=2)[:3000])
        except json.JSONDecodeError:
            print(item["text"][:3000])

In [ ]:
# Count records in an entity set
result = mcp.call_tool(
    "sap-target___odata_count",
    {
        "service_name": "API_BUSINESS_PARTNER",
        "entity_set": "A_BusinessPartner",
    },
)
print("=== Business Partner Count ===")
content = result.get("result", {}).get("content", [])
for item in content:
    if item.get("type") == "text":
        print(f"Total business partners: {item['text']}")

## Step 7: Use Strands Agent with SAP Tools

Connect a Strands Agent to the gateway and let it explore SAP data via natural language.

In [ ]:
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

mcp_client = MCPClient(
    lambda: streamablehttp_client(
        url=GATEWAY_URL,
        headers={
            "Authorization": f"Bearer {get_gw_token()}",
            "MCP-Protocol-Version": "2025-11-25",
        },
    )
)

SYSTEM_PROMPT = (
    "You are a helpful assistant with access to SAP tools via the AWS for SAP MCP Server. "
    "Before reading data, use get_metadata to understand available entity sets and field names. "
    "Use odata_count before odata_read to understand data volume. "
    "SAP field names can be non-obvious — always check metadata first."
)

with mcp_client:
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-6-v1",
        system_prompt=SYSTEM_PROMPT,
        tools=mcp_client.list_tools_sync(),
    )
    result = agent("What SAP services are available for sales orders? Show me the first 3 sales orders.")
    print(result)

## Best Practices

From the [AWS for SAP MCP Server documentation](https://docs.aws.amazon.com/mcp-sap/latest/awsforsapmcp/security.html):

1. **Read-only by default** — Write tools are disabled unless both global and per-operation flags are set to `true`. Only enable writes you actually need.
2. **Principle of least privilege** — Assign the SAP user only the minimum necessary roles and authorizations.
3. **Scope OAuth access** — Configure OAuth scopes to grant access only to specific required OData services.
4. **Use odata_count before reads** — Understand data volume before issuing broad reads to avoid overwhelming the agent.
5. **Use get_metadata proactively** — SAP field names are often non-obvious (e.g., `OverallOrdReltdBillgStatus` not `OverallBillingStatus`). Always check metadata before constructing filters.
6. **Credentials never stored on disk** — The MCP Server retrieves credentials at runtime from AWS Secrets Manager.

## Step 8: Clean Up

Delete resources created in this notebook. Skip if you plan to continue with [03-cross-isv-queries.ipynb](03-cross-isv-queries.ipynb).

In [ ]:
SKIP_CLEANUP = input("Skip cleanup to use gateway in notebook 03? (yes/no): ").strip().lower() == "yes"

if not SKIP_CLEANUP:
    # Delete SAP gateway target
    print("Deleting SAP gateway target...")
    gateway_client.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=SAP_TARGET_ID)
    print("  ✓ SAP target deleted")

    # Delete SAP credential provider
    print("Deleting SAP credential provider...")
    gateway_client.delete_oauth2_credential_provider(name=SAP_CREDENTIAL_PROVIDER_NAME)
    print("  ✓ Credential provider deleted")

    if CREATED_GATEWAY:
        # Delete gateway
        print("Deleting gateway...")
        gateway_client.delete_gateway(gatewayIdentifier=GATEWAY_ID)
        print("  ✓ Gateway deleted")

        # Delete Cognito resources
        print("Deleting Cognito resources...")
        cognito_client.delete_user_pool_domain(Domain=COGNITO_DOMAIN, UserPoolId=USER_POOL_ID)
        cognito_client.delete_user_pool(UserPoolId=USER_POOL_ID)
        print("  ✓ Cognito pool deleted")

        # Delete IAM role
        print("Deleting IAM role...")
        iam.delete_role(RoleName=ROLE_NAME)
        print("  ✓ IAM role deleted")

    print("\n✓ All resources cleaned up")
else:
    print("\nSkipping cleanup. Remember to clean up resources when done!")
    print(f"  Gateway ID: {GATEWAY_ID}")
    print(f"  Gateway URL: {GATEWAY_URL}")
    print(f"  SAP Target ID: {SAP_TARGET_ID}")